# 04_202 · Transformers planos con cuatro categorías

Reentrena MiniLM multilingüe y E5-small para cuatro daños sobre exactamente el mismo dataset 4:1 y splits usados por `04_205`. `SEGURO` se deriva y `ACOSO_AMENAZA` fusiona acoso personal con amenaza directa.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'cuadernos': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from scripts_auxiliares import entrenar_transformers_planos_4 as t4
from scripts_auxiliares import experimentos_jerarquicos_4 as h4
print('Dispositivo:', h4.device())
print('Modelos:', t4.MODEL_KEYS)
print('Objetivos:', t4.TARGET_LABELS)

ModuleNotFoundError: No module named 'scripts_auxiliares'

## 1. Contrato común de datos

El hash del dataset, manifiesto y checkpoints fuente se integra en un fingerprint. Si cualquiera cambia, un resultado anterior no se reutiliza silenciosamente.

In [ ]:
context = t4.load_context()
summary = t4.dataset_summary(context)
display(summary)
print('Dataset SHA-256:', context['dataset_sha256'])
print('Fingerprint:', context['training_fingerprint_sha256'])

In [ ]:
ax = summary.set_index('split')[t4.TARGET_LABELS].plot.bar(figsize=(11, 5))
ax.set_title('Positivos por Transformer y partición común')
ax.set_ylabel('Chunks positivos')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2. Transferencia y entrenamiento

Cada modelo parte de su checkpoint completo de `04_2`. Se copia el encoder adaptado al dominio; las filas de racismo, género y sexual se conservan y `ACOSO_AMENAZA` se inicializa con el promedio de las filas anteriores de acoso y amenaza. La cabeza y el encoder vuelven a optimizarse. Los umbrales históricos se descartan.

Se permiten hasta tres épocas con parada temprana según PR-AUC macro de validation. Se guardan `best_checkpoint`, `last_checkpoint`, historial por época, scores y reportes. Test no interviene en la elección de modelo o época.

In [ ]:
FORCE = False
result = t4.run_all(force=FORCE)
display(result['selection'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, key in zip(axes, t4.MODEL_KEYS):
    history = pd.DataFrame(result['models'][key]['history'])
    display(Markdown(f'### {key}'))
    display(history)
    history.plot(x='epoch', y=['validation_damage_pr_auc_macro', 'validation_damage_f1_macro', 'validation_any_damage_recall'], marker='o', ax=ax)
    ax.set_title(key)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Comparación final plana

In [ ]:
comparison = pd.read_csv(t4.METRICS_DIR / 'comparacion.csv')
display(comparison)
display(Image(filename=str(t4.FIGURES_DIR / 'comparacion_test.png')))
display(Markdown(
    f'**Resultado:** `{t4.RESULT_PATH.relative_to(ROOT)}`  \n'
    f'**Informe:** `{t4.REPORT_PATH.relative_to(ROOT)}`  \n'
    f'**Modelos:** `{t4.MODEL_DIR.relative_to(ROOT)}`'
))

## Referencias (APA 7)

Cawley, G. C., & Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research, 11*, 2079–2107. https://www.jmlr.org/papers/v11/cawley10a.html

Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE, 10*(3), e0118432. https://doi.org/10.1371/journal.pone.0118432